# Fine-tuning open models with agents: from eval to deployment

*Durable, self-healing agents on Union.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unionai/workshops/blob/factory/tutorials/langgraph-model-factory-agent/langgraph-model-factory-agent-tutorial.ipynb)
<!-- branch: switch blob/factory to blob/main once merged -->


There are two agents in this notebook.

The **support agent** is the one in production. A ticket comes in, it routes it to a queue, and the queue's drafting agent writes a reply. Today the routing step is an API call per ticket: accurate, slow, and billed. We are going to replace that call with a model we own, and we are not going to pick the model ourselves.

The **ML engineer agent** is a LangGraph agent that gets this request and operates the *model factory* on Union to fill it:

> Replace the support agent's router with a self-hosted open-source model: at least 95% accuracy on our tickets, under 150 ms per ticket on a T4. Baseline up to 5 candidates, fine-tune up to 3 of them, spend at most 12 runs.

| Step | You will | Minutes |
|---|---|---|
| 0 | Run the support agent as it is today. The "before". | 3 |
| 1 | Drive the factory yourself, no agent: parallel GPU evals, one fine-tune. | 5 |
| 2 | Hand the request to the ML engineer agent. It builds, deploys and tests the router. | 5 |
| 3 | Switch the support agent to the new router. The "after". | 3 |
| 4 | Drift: a new kind of ticket arrives. See it. | 3 |
| 5 | The loop closes itself: publish the tickets, a trigger notices, the engineer retrains. | 10 |
| 6 | Bake off the engineer's brain, including an open model (optional). | 5 |
| 7 | Kill the engineer mid-run and watch it resume (optional). | 5 |

Every step is a Flyte task. On a cluster, `run(task)` sends it there, prints the run URL, waits, and returns the result. By default (`LOCAL = True` in the setup cell) the same `run(task)` executes it in this kernel, on Colab's GPU, with no devbox and no cluster: steps 0 to 4 and 6 work that way, with a folder standing in for the artifact registry and the app. With a Union cluster, set `LOCAL = False` and connect below: that is the full tutorial, including step 5's triggers and step 7's crash. On a cluster, **open every run URL**: the run graph and the reports are the point; in local mode the step's report renders under its cell, and `show_reports(all=True)` shows every tool's report from that run (each eval, each fine-tune). The `view_file(...)` cells show the code behind each step; read them when you want to know how, skip them when you want to keep moving.

---

## Setup

Pick where the steps run (`LOCAL`), then run the next cell once. In a hosted notebook (SageMaker, Colab, or any kernel that opened this file on its own) it clones the repo, moves into the tutorial folder, installs the dependencies (a minute or two), and sets up a plain-text keyring so the cluster login can store its token. Locally, with the repo cloned and a virtualenv from `requirements.txt`, it only does the imports.

Two helpers come out of it: `run`, which submits a task to the cluster and waits (or runs it here, in local mode), and `view_file`, which renders a source file inline.


In [1]:
import os
import platform
import sys
from pathlib import Path

# Where the steps run. True (the default): in this kernel, on Colab's T4 (or a laptop with no
# devbox), with a local folder standing in for the registry and the app; steps 0-4 and 6 work.
# False: on a Union cluster (the next section connects to it), with the parallel GPUs, the
# artifact registry, the deployed app and the triggers; everything works, including 5 and 7.
LOCAL = True

HOSTED = not Path("utils/workshop.py").exists()  # not inside the tutorial folder: a hosted notebook
if HOSTED:
    !git clone -q -b factory https://github.com/unionai/workshops  # branch: switch to main once merged
    %cd workshops/tutorials/langgraph-model-factory-agent
    REQS = "requirements-local.txt" if LOCAL else "requirements.txt"  # local mode also needs the training stack
    !{sys.executable} -m pip install -q -r {REQS} keyrings.alt
    %env TERM=dumb
if platform.system() == "Linux" and not Path.home().joinpath(".config/python_keyring/keyringrc.cfg").exists():
    # No system keyring on a hosted Linux kernel: store the login token in a plain file instead.
    !mkdir -p ~/.config/python_keyring && printf '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring\n' > ~/.config/python_keyring/keyringrc.cfg

from getpass import getpass
from utils.file_viewer import view_file
from utils.workshop import run, show, show_reports

if LOCAL:
    from functools import partial

    run = partial(run, local=True)  # every run(...) below now executes in this kernel


### Connect to the cluster

*Local mode (the default): nothing to connect to; the next two cells do nothing. Set `LOCAL = False` above to use a cluster.*

You were given a Union endpoint and a project. This writes the config the SDK and the `flyte` CLI read. The first cluster call after it prints a login URL and a code: open the URL, sign in, paste the code back into the cell. A hosted notebook has no browser to hand off to, which is why the login is `headless`.

Everyone in the room shares this project, and one GPU pool. The engineer asks for several evals in one turn and each takes a T4, so if runs start piling up in "waiting for resources", set `FACTORY_MAX_PARALLEL_GPU` in the next cell (the host will say): the calls still go out together, they just take turns. Fine-tunes are cached per project, so after the first person trains a model nobody trains it again; evals run every time, so every run shows its work. The serving app and the artifacts are shared too, which means the last promotion wins. If you would rather have your own, set `os.environ["FACTORY_TAG"] = "yourname"` before the next cell and everything you deploy carries that suffix.

In [ ]:
ENDPOINT = "tryv2.hosted.unionai.cloud"
PROJECT = "factorysea"

# In a full room, cap how many T4s one run may hold at once (the calls still go out together, they queue).
# os.environ["FACTORY_MAX_PARALLEL_GPU"] = "2"

if not LOCAL:
    !flyte create config --endpoint {ENDPOINT} --project {PROJECT} --domain development --builder remote --auth-type headless --force


### The agent's model (optional)

Both agents think with Claude Haiku 4.5 by default: the same routing accuracy as Opus on our tickets, four times faster, a ninth of the cost. Any of these work, and the whole tutorial is the same with any of them; only the numbers move:

| `AGENT_MODEL` | Key it needs on the cluster | Why you would pick it |
|---|---|---|
| `anthropic:claude-haiku-4-5` (default) | `ANTHROPIC_API_KEY` | Fast, cheap, meets the request every time we ran it |
| `anthropic:claude-opus-5` | `ANTHROPIC_API_KEY` | The big model. Thinks before it acts; better rationales, four times slower, nine times the cost |
| `openai:gpt-4.1` | `OPENAI_API_KEY` | Met the request in the fewest runs (7) in our bake-off |
| `openai:gpt-4.1-mini` | `OPENAI_API_KEY` | Cheapest API option; needed 11 runs |
| `vllm:qwen3-8b` | `VLLM_API_KEY`, plus `VLLM_BASE_URL` | An open model you serve yourself (see step 6) |

The key lives on the cluster as a secret named after the provider, so nothing is needed here for cluster runs with the default. To switch, set `AGENT_MODEL` and `FACTORY_PROVIDERS` before running anything, or pass `model=...` to a single `run(...)` call. A local run (`run(task, local=True)`) needs the key in this process.


In [ ]:
# Pick the model that drives both agents (see the table above). The default needs nothing.
# os.environ["AGENT_MODEL"] = "anthropic:claude-opus-5"
# os.environ["AGENT_MODEL"] = "openai:gpt-4.1"
# os.environ["AGENT_MODEL"] = "openai:gpt-4.1-mini"
# os.environ["FACTORY_PROVIDERS"] = "openai"          # the providers whose secrets every agent task asks for

# Local runs read the key from this kernel's environment; cluster runs read the secret (next cell).
if LOCAL and not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
# if LOCAL and not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")


### The API key on the cluster

Tasks read the key from a Union secret named after the provider: `ANTHROPIC_API_KEY`, or `OPENAI_API_KEY`. The workshop project already has one. If you are on your own cluster, create it once (the `--value` flag matters: without it the command waits for input that never comes in a notebook).

In [ ]:
# !flyte create secret ANTHROPIC_API_KEY --value sk-ant-... --project {PROJECT} --domain development
if not LOCAL:
    !flyte get secret --project {PROJECT} --domain development


---

## 0. The support agent, as it is today

**Files:** `support_agent.py` (the agent: route, then draft), `tickets.py` (the synthetic tickets and queues), `llm.py` (the model, from a string), `utils/workshop.py` (`run`)

Thirty held-out tickets go through the support agent: the router picks a queue, then the queue's drafting agent (the same model, prompted as that team) writes a two-line reply. Open the run and read the report: routing accuracy, p50 route latency, tokens, and cost per 1,000 tickets. This is the "before".

What we measured on the same thirty tickets, so you know what to expect (cost is routing plus replies, at list price):

| Support agent model | Routing accuracy | Route p50 | Per 1,000 tickets |
|---|---|---|---|
| Claude Opus 5 | 96.7% | 2.28 s | \$10.28 |
| Claude Haiku 4.5 (the default) | 96.7% | 596 ms | \$1.18 |
| GPT-4.1 | 93.3% | 594 ms | \$0.64 |

Opus thinks before it routes, so it is slower at the same accuracy. To see it anyway, pass `model="anthropic:claude-opus-5"` to `run`.

In [ ]:
from support_agent import agent_handle_tickets

before = run(agent_handle_tickets, n=30, router="llm")
show(before, ["routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd", "tokens"])

**What to look at.** In the run graph, every ticket is a group of its own, `ticket T-11-oth-008`, holding two traced steps: `route_with_model` and `draft_reply`. Each ticket is a small agent of its own, router then responder, and the groups keep the thirty of them apart. Click one: the inputs are the ticket text and the model name, the output is the queue or the reply. Because each step is recorded, a batch that dies halfway replays what it already did instead of paying for it twice.

**How it is built.** Two LangGraph nodes, `route → draft`, in a single Flyte task. `router="llm"` is today's routing, an API call per ticket. `router="oss"` calls the app we are about to build.

In [ ]:
view_file("support_agent.py", show_path=True)

---

## 1. The factory, with you at the controls

**Files:** `model_factory.py` (this step), `factory.py` (`evaluate()` and `fine_tune()`, no Flyte in it), `tools.py` (the same functions as Flyte tasks with GPUs, caching and reports), `report.py` (the HTML)

Before the engineer touches anything, drive the factory yourself. This one task baselines three candidate models on 120 held-out tickets, as three T4 containers running at the same time, then fine-tunes one of them for an epoch and evaluates it again.

Open the run while it works. The three evals sit side by side under `baseline-evals`, each named after its model (`run_eval · qwen2.5-0.5b`), each with its own report: the numbers, a per-category table, and a confusion matrix that shows where the misroutes went. The fine-tune under `fine-tune-and-recheck` draws its loss curve live. The parent report is the eval table.

The fine-tune is cached by its inputs: when the engineer asks for the same one in step 2, it gets the weights back in seconds, and so does everyone else in the room. The evals are not cached, on purpose, so the engineer measures again and you watch it happen.

In [ ]:
from model_factory import model_factory

run(model_factory, fine_tune_too=True)

In [ ]:
if LOCAL:
    # every report of that run, oldest first: each eval's table and confusion matrix, each
    # fine-tune's loss curve, and the step's own report last. A fine-tune that hit the cache
    # (same model, data, and epochs as an earlier run) did not run again, so it has no report.
    show_reports(all=True)


**How it is built.** `factory.py` is plain transformers, peft and trl: `evaluate()` and `fine_tune()`, no Flyte in it. `tools.py` wraps those functions as Flyte tasks with the GPU, the cache, the retries and the reports, and those tasks are the tools the engineer gets. The split is deliberate, not something LangGraph needs. A tool's docstring is a prompt (the model reads it to decide when to call it), and `evaluate()`'s is for engineers. One ML function backs several tasks: `evaluate()` runs on the T4 env, the CPU env and the validation trigger without changing. And `factory.py` needs nothing but torch, so it runs in a test, a Colab GPU, or the serving app. If you have written Flyte pipelines by hand, this is the same idea with the composition moved: step 1 composes the tasks in code like a pipeline; step 2 hands the same tasks to an agent and lets it compose them at run time.

In [ ]:
view_file("factory.py", show_path=True)

In [ ]:
view_file("tools.py", show_path=True)

---

## 2. Hand the request to the ML engineer agent

**Files:** `ml_engineer.py` (this step), `graph.py` (the request, the graph, the typed decision), `tools.py` (the eight tools), `router_app.py` (what `promote` deploys)

Now the same factory, operated by an agent. The engineer's graph is `think → tools → think … → decision`: the model reads the request, calls tools, reads the results, and keeps going until it has something to promote or has spent the budget. Every model turn is a recorded step; every tool call is a durable child action on its own container. Independent calls it asks for in one turn run in parallel, so "evaluate these four" is four T4s at once.

Open the run while it works. Every action is named after what it does (`run_eval · qwen2.5-0.5b`, `fine_tune · modernbert-base · 3ep`). Each `think:model` step shows the turn number and the message the model is reacting to as inputs, and what it said and asked for as outputs. At the end, `promote` publishes the winner as the `ticket-router` artifact and deploys the serving app, sized to the model it chose, and `test_deployment` calls the live app with tickets it has not seen. The report has a **Graph** tab with the LangGraph graph drawn, up before the first turn, and the decision page as its main tab: what was promoted, pass or fail against the request, everything it measured, the sequence of calls, and the rationale in the agent's own words.

This takes five to ten minutes when the cluster is warm; most of it is the evals and fine-tunes on T4s, which you can watch in the run graph. If the GPU nodes have to boot first, add a few minutes to the first evals. The last step, `promote`, looks quiet for a while: it is waiting for the new serving pod to take over from the old one before it lets `test_deployment` measure anything. That rollout is a minute or two on most clusters, but on the demo cluster it has taken 10 to 15 minutes, so a 25-minute run is not stuck.

*Local mode:* `promote` has no registry or app to go to. It copies the winner into `.factory/ticket-router/<version>/` next to the code and points `current.json` at it; `test_deployment` loads that in-process, and so does the support agent in the next step.


In [ ]:
from ml_engineer import ml_engineer_agent

decision = run(ml_engineer_agent)
show(decision["decision"], ["action", "promoted_model", "accuracy", "latency_p50_ms", "deployment_test_passed", "runs_spent"])

In [ ]:
if LOCAL:
    # every report of that run, oldest first: each eval's table and confusion matrix, each
    # fine-tune's loss curve, and the step's own report last. A fine-tune that hit the cache
    # (same model, data, and epochs as an earlier run) did not run again, so it has no report.
    show_reports(all=True)


### The answer key: every candidate, measured

Your engineer just produced most of these numbers itself; put its decision next to the whole matrix. Five candidates, all Apache-2.0, from 149M to 1.7B parameters, measured on the same 120 tickets, on a T4:

| candidate | zero-shot | after fine-tune | p50 on T4 |
|---|---|---|---|
| smollm2-360m | 13% | | 110 ms |
| qwen2.5-0.5b | 59% | 1 epoch: 97.5% | 68 ms |
| qwen2.5-1.5b | 79% | 1 epoch: 97.5% | 83 ms |
| smollm2-1.7b | 36% | | 72 ms |
| modernbert-base (an encoder, not a chat model) | 9%, untrained head | 3 epochs, 16 s: 99.2% | 14 ms |

Nothing passes zero-shot. The bar is 95%. The engineer had to work that out from numbers it produced itself, and notice that the one model that is not a chat model at all turns out to be the best router once it is trained. If yours promoted the encoder, it read the matrix right; if it promoted the 0.5B or 1.5B, look at its rationale for why it thought the extra size was worth it.

For scale, the models the support agent could route with today, zero-shot on the same 120 tickets (p50 is a network call from the cluster):

| router | zero-shot | p50 | per 1,000 routes | where it runs |
|---|---|---|---|---|
| Claude Opus 5 | 98.3% | 1.7 s | \$3.81 | API |
| GPT-4.1 | 97.5% | 634 ms | \$0.28 | API |
| Claude Haiku 4.5 | 95.8% | 606 ms | \$0.89 | API |
| Qwen3-8B (vLLM) | 94.2% | 386 ms | ~\$0.20 | one L40s, ours |
| modernbert-base, fine-tuned | 99.2% | 14 ms on a T4 | < \$0.01 | 149M params, ours |

The big API models clear the bar without training; the trained encoder beats all of them at a hundredth of the latency. Cost is routing only, at list prices, measured from the tokens each call used (the Claude calls carry the tool-use schema, GPT does not). At 100k tickets a month that is \$381 on Opus, \$89 on Haiku, and under a dollar of CPU time on the encoder; the accuracy gap between them is one ticket in 120 the other way.

**What just happened.** The agent had eight tools and a budget of twelve runs. It baselined the candidates, found nothing that passed, fine-tuned the ones worth it (in parallel), picked the smallest model that cleared both bars, promoted it, and tested the deployment. Nothing in that sequence was scripted; the order and the choices are in the report.

**A person in the loop, if you want one.** Here the agent promotes on its own. If nothing should reach production without a sign-off, set `os.environ["FACTORY_APPROVAL"] = "1"` before the setup imports and run this step again: `promote` then pauses the run on a Flyte condition, the Union UI shows the model, its numbers and the agent's reasoning with an approve button, and the run continues when someone clicks it (or declines, in which case the agent is told and writes that up). No extra infrastructure: it is a native Flyte feature, and the same gate applies wherever `promote` runs, including the trigger-started retrain in step 5.

**How it is built.** `graph.py` holds the request, the typed `Decision` the run ends with, the parallel tool node, and the `think` step. The tools are the tasks from `tools.py`. The scoring in the report checks the agent's claims against its own measurements, because an agent that says "94.2% rounds to 95" is a thing that happens.

In [ ]:
view_file("graph.py", show_path=True)

---

## 3. Switch the support agent to the new router

**Files:** `support_agent.py` (`router="oss"`), `router_app.py` (the serving app it calls)

Same thirty tickets, same draft replies, but routing is now an HTTP call to the app the engineer deployed. Nothing in the support agent changed except a flag. Put this report next to step 0's: accuracy holds or improves, latency drops by an order of magnitude, and the cost that remains is the reply drafts. The cost tile counts API tokens only: the router pod is not metered, and at 2 CPUs (a few cents an hour on any cloud) it adds about a tenth of a cent per thousand tickets.

What we measured, before → after:

| Support agent model | Routing accuracy | Route p50 | Per 1,000 tickets |
|---|---|---|---|
| Claude Opus 5 | 96.7% → 100% | 2.28 s → 128 ms | \$10.28 → \$6.47 |
| Claude Haiku 4.5 | 96.7% → 100% | 596 ms → 170 ms | \$1.18 → \$0.30 |
| GPT-4.1 | 93.3% → 96.7% | 594 ms → 200 ms | \$0.64 → \$0.34 |

*Local mode:* there is no app. `router="oss"` loads the promoted model in this process, so the route latency is the model on this kernel's GPU rather than a pod, and the comparison still holds.


In [ ]:
after = run(agent_handle_tickets, n=30, router="oss")
show({"before": {k: before[k] for k in ("routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd")},
      "after":  {k: after[k]  for k in ("routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd", "mean_confidence")}})


**How it is built.** The router app is a FastAPI service on Union that mounts the promoted artifact at startup and classifies on `/classify`. The pod is sized to the model: a small CPU pod for the encoder, a T4 for a larger chat model, so the latency the request was judged on is the latency production sees.

In [ ]:
view_file("router_app.py", show_path=True)

---

## 4. Drift: a new kind of ticket

**Files:** `support_agent.py` (`dataset="v2"`), `tickets.py` (`DATASET_VERSIONS`: v2 adds `data_request`)

The support team adds a ninth queue, `data_request`, for GDPR-style tickets ("delete my account and all my data", "send me everything you have on me"). Your router has never seen the label. See it first: the whole v2 test split through the support agent, still routing with the model deployed in step 2.

Two things to read in the report. First, where the GDPR tickets went: not to `other`, but mostly to `account_access`, because "delete my account and all my data" looks like an account ticket to a model that only knows eight queues. Second, the confidence column next to them. On our runs, a third to a half of the misrouted GDPR tickets carried 90%+ confidence, depending on the fine-tune, and the batch's mean confidence barely moved from the v1 batch in step 3. A classifier does not know what it does not know, so watching its confidence would not have raised this alarm.

What catches it is labels. Here that is the answer key; in production it is the support team re-routing tickets by hand, which is the same information a day late. That is why the next step starts from people publishing labeled tickets, not from a metric. The report says all of this in one paragraph under the tiles.


In [ ]:
drift = run(agent_handle_tickets, n=126, router="oss", dataset="v2")
show(drift, ["routing_accuracy", "mean_confidence", "low_confidence_share", "misroutes", "confidently_wrong",
             "tickets_with_unknown_queue", "queue_distribution"])


---

## 5. The loop closes itself

*Local mode: triggers and artifacts are the platform's. Skip the two cells below and run the by-hand version further down; the after-fix cell then works as on the cluster.*

**Files:** `adaptive_loop.py` (`publish_tickets`, the trigger, `adapt`), `validate_on_promote.py` (the validation trigger), `drift.py` (the fix by hand)

Somebody did notice: the support team saw GDPR tickets in the wrong queue, added the `data_request` queue, and labeled the batch. From here on nobody should have to do anything. Deploy two triggers once. Deploying registers a task together with the trigger that starts it, so from here on the platform, not you, launches these runs. The first watches the `support-tickets` artifact: whenever a new version lands, a run of `adapt` starts that you did not start. It is the same engineer, told "new tickets landed, check production first". It measures the live model on the new tickets, finds it below the bar, retrains on them, promotes, and tests the deployment. That promotion fires the second trigger, which validates every new `ticket-router` version on a T4, whoever published it. If production had still met the bar, `adapt` would have kept it and stopped. And "nobody clicked" is a choice, not a requirement: with `FACTORY_APPROVAL=1` set before the triggers are deployed, the retrain still starts on its own but pauses at `promote` until a person approves the new model in the Union UI. One click, with the numbers in front of you, is often the right amount of human in the loop for a production model.

In [ ]:
if not LOCAL:
    !flyte deploy adaptive_loop.py agent_env
    !flyte deploy validate_on_promote.py validate_env


**How it is built.** A trigger is a few lines next to the task it starts. `adaptive_loop.py` declares one with `flyte.Trigger`: its automation is `flyte.OnArtifact("support-tickets")`, and its inputs map the task's `tickets` argument to `flyte.TriggeredArtifact`, the version that just landed. The task decorator lists it in `triggers=(...)`, and `flyte deploy` registers both. `validate_on_promote.py` is the same pattern on the `ticket-router` artifact, which is why it fires on every promotion whether an agent or a person made it.

Two things to know. A trigger pins the task version it was deployed with, so after you change the code, deploy again or the trigger keeps running the old version. And the artifact version has to be new each time: publishing the same version twice does not fire anything, which is why `publish_tickets` stamps every publish with a fresh version.

In [ ]:
view_file("adaptive_loop.py", show_path=True)

Now publish the v2 tickets. Then open the runs list in the Flyte UI and watch: a run of `adapt` appears on its own (a minute or so), then a run of `validate_router`. Each firing also leaves a run named `at-<trigger>` with no inputs or report: that is the trigger's own record, and the work is in the task run next to it. Open the `adapt` run: it is a full decision page, made without you.

In [ ]:
if not LOCAL:
    from adaptive_loop import publish_tickets

    run(publish_tickets, version="v2")
    print("watch the runs list: 'adapt' appears on its own, then 'validate_router'")


### The same fix, by hand

If you would rather watch the engineer do it in front of you, this is the same request without the trigger: tell the engineer what changed and let it work. In local mode this is the only way, and the cell runs it.


In [ ]:
by_hand = LOCAL  # True: the engineer does the fix here, told what changed, instead of the trigger
if by_hand:
    from drift import day_two

    fixed = run(day_two)
    show(fixed["decision"], ["action", "promoted_model", "accuracy", "deployment_test_passed", "runs_spent"])


When the `adapt` run has finished (or the by-hand fix above), the support agent is fine again. The `ticket-router` artifact has a new version; open it in the UI and look at Versions and Lineage: from the app, back through the promotion, to the fine-tune, to the tickets that caused it.

In [ ]:
after_fix = run(agent_handle_tickets, n=126, router="oss", dataset="v2", labels_from="v2")
show(after_fix, ["routing_accuracy", "mean_confidence", "tickets_with_unknown_queue", "queue_distribution"])


---

## 6. Bake off the engineer's brain (optional)

**Files:** `bakeoff.py` (this step), `llm.py` (the model strings), `serve_model.py` (the open model on vLLM)

The engineer's own model is a model too, and it can be graded like any other. Same request, several models driving the engineer in parallel, scored on whether they promoted something that meets it and how many runs they spent. Promotion publishes artifacts here but does not redeploy the router. Skip if short on time.

One contender can be an open model: Qwen3-8B served by vLLM on a Union app with an L40s (`python serve_model.py` deploys it once for a whole room). On our cluster it clears the bar too, promoting the fine-tuned 1.5B at 100% and 84 ms while using all 12 runs, so the ML engineer can be off the API as well, and the whole loop, brain included, runs on models you own.

In [ ]:
# from bakeoff import bakeoff
# run(bakeoff, models=["anthropic:claude-haiku-4-5", "anthropic:claude-opus-5"])

# With an open model served as an app (serve_model.py on Union, serve_model_apple.py on a Mac devbox), add it.
# Knobs like FACTORY_PROVIDERS are read when the tutorial modules are imported (step 0), so set these from the
# shell for this one run rather than with os.environ here:
# !VLLM_BASE_URL=https://<the vllm app>/v1 VLLM_API_KEY_SECRET_NAME=VLLM_API_KEY FACTORY_PROVIDERS=anthropic,vllm \
#     flyte run bakeoff.py bakeoff --models '["anthropic:claude-haiku-4-5", "vllm:qwen3-8b"]'


---

## 7. Kill the engineer, watch it resume (optional)

*Cluster only: the retry in a fresh container and the replay of recorded model turns are the platform's.*

**Files:** `crash_resume.py` (the booby-trapped model and the retry)

A durable run is not one process. This is the same engineer with a booby-trapped model: it dies after its third live model call on the first attempt, right after the fine-tunes come back. Flyte retries it in a fresh container, where the model turns it already paid for replay from their records, the evals and fine-tunes are cache hits, and the decision is produced once.

In the run graph, expect exactly one red `think:model`: that is the crash itself, with the message `simulated worker crash after 3 model calls`. Everything after it is attempt 1. In the pod logs, attempt 0 prints three `live model call` lines and the crash; attempt 1 prints four.

In [ ]:
# from crash_resume import resilient_engineer
# run(resilient_engineer)

---

## 8. Stretch goals: a person approves, and an open model does the engineering

**Files:** `tools.py` (`_ask_for_approval`, in `promote`), `serve_model_apple.py` / `serve_model.py` (the open model as an app), `llm.py`

Two optional runs, each a plain `flyte run` from the shell so the knob is set for the whole run. Both need a cluster or a devbox; a local run has no conditions and no app.

**A person approves the deploy.** With `FACTORY_APPROVAL=1`, `promote` creates a `flyte.new_condition` before it publishes anything. The run pauses; in the Flyte UI the run shows the condition with the model, its numbers and the agent's reasoning, and an approve button. The cell below blocks until someone decides. Approve from the UI, or from a terminal with `flyte signal condition <run> <action> true` (`false` declines, and the agent writes up that it was refused). The same gate applies wherever `promote` runs, including the trigger-started retrain in step 5.

**The engineer's brain is an open model.** Step 6 showed Qwen3-8B as one contender; here it is the only brain, so the whole loop runs on models you own: the factory, the router and the engineer. On a Union cluster `python serve_model.py` serves it on vLLM; on an Apple Silicon devbox `python serve_model_apple.py` serves it on the Mac's GPU. Put the printed endpoint in `VLLM_BASE_URL`. It meets the request (10 to 12 runs, about 35k tokens); it is slower than the API models and tends to pick a bigger router than it needs, which the report will show you.


In [ ]:
if not LOCAL:
    # 1. The gate: the run pauses at promote until you approve it in the Flyte UI (or `flyte signal condition`).
    !FACTORY_APPROVAL=1 flyte run ml_engineer.py ml_engineer_agent --candidates_to_screen 2 --max_fine_tunes 1

    # 2. The engineer on Qwen3-8B, no API key: fill in the endpoint printed by serve_model.py / serve_model_apple.py.
    # !VLLM_BASE_URL=http://<endpoint>/v1 FACTORY_PROVIDERS=vllm flyte run ml_engineer.py ml_engineer_agent --model vllm:qwen3-8b


---

## Where to go from here

- Change the request: `run(ml_engineer_agent, min_accuracy=0.98, max_latency_ms=50)` is a different problem.
- Add a candidate to `factory.CANDIDATES`. Anything on the Hub with a chat template, or any encoder, works.
- Replace `tickets.py` with your own data. The tools do not care where the tickets come from.
- Detect drift without waiting for labels. Step 4 showed that the classifier's own confidence is a weak alarm for a new kind of ticket. Three signals work better: the share of tickets the support team re-routes by hand (free labels, with a lag); a novelty score from the encoder's embeddings against the training set; and an LLM judge, the step 0 router shadowing a sample of routes with a "none of these queues" option the softmax cannot express. Any of them can publish `support-tickets` and start the loop in step 5.
- Turn on the approval gate: `os.environ["FACTORY_APPROVAL"] = "1"` and `promote` pauses in the Flyte UI until someone says yes.
- The ticket router is one example. The same loop works for a reranker, a fraud scorer, a PII detector, or an intent model: anywhere an agent leans on a frontier model for a narrow job and you have the data.
- The README has the full write-up, the timings, and the gotchas we hit building this.